# Notebook 03: Mask Fine-Tuning with Classification Heads

In this notebook, we perform end-to-end training but **freeze the entire Moirai encoder except for the mask tokens**. 

In [25]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import copy
from IPython.display import display

from tslearn.datasets import UCR_UEA_datasets
from sklearn.preprocessing import LabelEncoder
from uni2ts.model.moirai import MoiraiModule
from encoder import MoiraiEncoder

from heads import (
    MeanPoolingClassifier, 
    SingleScaleAttentionClassifier, 
    SingleScaleMultiHeadClassifier, 
    HierarchicalMultiHeadClassifier
)
from models import MoiraiClassifier, MoiraiMaskTuner
from utils import preprocess_data, create_raw_dataloaders, train_finetune

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES = [16]#[8, 16, 32, 64]
BATCH_SIZE = 32

In [26]:
def unfreeze_only_moirai_mask(encoder):
    for param in encoder.parameters():
        param.requires_grad = False
    for name, param in encoder.named_parameters():
        if "mask" in name.lower():
            param.requires_grad = True

def grid_search_finetune(model_class, model_kwargs, train_loader, val_loader, test_loader, device="cuda"):
    lr_grid = [1e-4]
    wd_grid = [0.05]
    best_val_loss = float('inf')
    best_model = None
    
    for wd in wd_grid:
        for lr in lr_grid:
            model = model_class(**model_kwargs).to(device)
            val_loss, trained_model = train_finetune(
                model=model, train_loader=train_loader, val_loader=val_loader,
                lr=lr, epochs=1000, weight_decay=wd, device=device
            )
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model = copy.deepcopy(trained_model)     
            
    best_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for b_t, b_o, b_p, b_y in test_loader:
            b_t, b_o, b_p, b_y = b_t.to(device), b_o.to(device), b_p.to(device), b_y.to(device)
            preds = torch.argmax(best_model(b_t, b_o, b_p), dim=-1)
            correct += (preds == b_y).sum().item()
            total += b_y.size(0)
            
    return best_model, correct / total

class MaskOnlyFinetunerWrapper(nn.Module):
    def __init__(self, patch_size, num_vars, num_classes, size="small"):
        super().__init__()
        moirai_enc = MoiraiEncoder(
            module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{size}"),
            prediction_length=patch_size, context_length=36, patch_size=patch_size, 
            num_samples=100, target_dim=num_vars, feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
        )
        unfreeze_only_moirai_mask(moirai_enc)
        self.model = MoiraiMaskTuner(encoder=moirai_enc, num_vars=num_vars, num_classes=num_classes)

    def forward(self, t, o, p):
        return self.model(t, o, p)

class HeadFinetunerWrapper(nn.Module):
    def __init__(self, head_class, head_kwargs, patch_size, num_vars, size="small", remove_last_patch=False):
        super().__init__()
        moirai_enc = MoiraiEncoder(
            module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{size}"),
            prediction_length=patch_size, context_length=36, patch_size=patch_size, 
            num_samples=100, target_dim=num_vars, feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
        )
        unfreeze_only_moirai_mask(moirai_enc)
        head = head_class(**head_kwargs)
        self.model = MoiraiClassifier(encoder=moirai_enc, head=head, remove_last_patch=remove_last_patch, num_vars=num_vars)

    def forward(self, t, o, p):
        return self.model(t, o, p)

In [27]:
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)
num_classes = len(set(y_train_encoded))

X_tr_t, X_tr_o, X_tr_p = preprocess_data(X_train, device=DEVICE)
X_te_t, X_te_o, X_te_p = preprocess_data(X_test, device=DEVICE)

y_tr_t = torch.tensor(y_train_encoded, dtype=torch.long, device=DEVICE)
y_te_t = torch.tensor(y_test_encoded, dtype=torch.long, device=DEVICE)

tr_loader, va_loader = create_raw_dataloaders(X_tr_t, X_tr_o, X_tr_p, y_tr_t, batch_size=BATCH_SIZE, device=DEVICE)
te_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_te_t, X_te_o, X_te_p, y_te_t), batch_size=BATCH_SIZE, shuffle=False
)

## 1. Baseline: Linear Model on Mask Embedding Only

In [28]:
df_mask_only = pd.DataFrame(index=["Mask Only"], columns=PATCH_SIZES)
df_mask_only.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    print(patch)
    _, test_acc = grid_search_finetune(
        model_class=MaskOnlyFinetunerWrapper,
        model_kwargs={"patch_size": patch, "num_vars": NUM_VARS, "num_classes": num_classes, "size": SIZE},
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
    )
    df_mask_only.loc["Mask Only", patch] = test_acc

display(df_mask_only.astype(float).round(4))

16
 Early stopping : 74


Patch Size,16
Mask Only,0.5856


## 2. Baseline: Mean Pooling on Full Sequence (Context + Mask)
We average all patches (including the unfrozen mask) before passing them to a linear classifier.

In [29]:
df_mean_pool = pd.DataFrame(index=["Mean Pooling"], columns=PATCH_SIZES)
df_mean_pool.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    _, test_acc = grid_search_finetune(
        model_class=HeadFinetunerWrapper,
        model_kwargs={
            "head_class": MeanPoolingClassifier,
            "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
            "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
    )
    df_mean_pool.loc["Mean Pooling", patch] = test_acc

display(df_mean_pool.astype(float).round(4))

 Early stopping : 140


Patch Size,16
Mean Pooling,0.6233


## 3. Advanced Pooling: Attention & MHA (Context + Mask)
We use Attention mechanisms to dynamically weight the patches. The network can now learn to pay attention to specific context patches AND the fine-tuned mask patch.

In [ ]:
PATCH_SIZES_TO_TEST = [8, 16] 
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 16

model_names_single = ["Attention (Base)", f"MHA (Heads={NUM_HEADS})"]
index_single = pd.MultiIndex.from_product([model_names_single, MODES], names=["Model", "Mode"])
df_adv_single = pd.DataFrame(index=index_single, columns=PATCH_SIZES_TO_TEST)
df_adv_single.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        # 1. Basic Attention
        _, acc_attn = grid_search_finetune(
            model_class=HeadFinetunerWrapper,
            model_kwargs={
                "head_class": SingleScaleAttentionClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        df_adv_single.loc[(model_names_single[0], mode), patch] = acc_attn

        # 2. Multi-Head Attention
        _, acc_mha = grid_search_finetune(
            model_class=HeadFinetunerWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        df_adv_single.loc[(model_names_single[1], mode), patch] = acc_mha

 Early stopping : 108
 Early stopping : 27
 Early stopping : 96
 Early stopping : 34
 Early stopping : 80
 Early stopping : 24
 Early stopping : 81
 Early stopping : 31


Patch Size                                8       16
Model            Mode                               
Attention (Base) shared_context       0.5949  0.5848
                 independent_context  0.6087  0.5981
MHA (Heads=16)   shared_context       0.6071  0.6188
                 independent_context  0.6091  0.6265

In [31]:
df_adv_single.astype(float).round(4)

Patch Size                                8       16
Model            Mode                               
Attention (Base) shared_context       0.5949  0.5848
                 independent_context  0.6087  0.5981
MHA (Heads=16)   shared_context       0.6071  0.6188
                 independent_context  0.6091  0.6265

## 4. Ablation Study: Number of Attention Heads
Testing the impact of `num_heads` on the Single-Scale MHA model (with mask fine-tuning) for a fixed patch size of 16.

In [ ]:
PATCH = 16
MODES = ["shared_context", "independent_context"]
HEADS_TO_TEST = [16, 32, 64, 128] 

df_heads_ablation = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_ablation.index.name = "Mode"
df_heads_ablation.columns.name = f"Num Heads (Patch {PATCH})"

for mode in MODES:
    for heads in HEADS_TO_TEST:
        
        _, test_acc = grid_search_finetune(
            model_class=HeadFinetunerWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {
                    "num_vars": NUM_VARS, 
                    "num_classes": num_classes, 
                    "patch_mode": mode, 
                    "num_heads": heads
                },
                "patch_size": PATCH, 
                "num_vars": NUM_VARS, 
                "size": SIZE, 
                "remove_last_patch": False
            },
            train_loader=tr_loader, 
            val_loader=va_loader, 
            test_loader=te_loader, 
            device=DEVICE
        )
        
        df_heads_ablation.loc[mode, heads] = test_acc

display(df_heads_ablation.astype(float).round(4))

 Early stopping : 23
 Early stopping : 24
 Early stopping : 28
 Early stopping : 26
 Early stopping : 30


# 5. Hierarchical

In [32]:
df_hierarchical = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_hierarchical.index.name = "Mode"
df_hierarchical.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        _, acc_hier = grid_search_finetune(
            model_class=HeadFinetunerWrapper,
            model_kwargs={
                "head_class": HierarchicalMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE
        )
        df_hierarchical.loc[mode, patch] = acc_hier

display(df_hierarchical.astype(float).round(4))

 Early stopping : 32
 Early stopping : 43
 Early stopping : 22
 Early stopping : 26


Patch Size,8,16
Mode,,
shared_context,0.5746,0.5795
independent_context,0.6046,0.5864
